# Clover Cancer — Gemma 4 Fine-tuning

Fine-tunes Google's Gemma 4 E2B model on pancreatic cancer triage data using Unsloth.

**Hardware:** Kaggle T4 GPU (16GB VRAM)  
**Author:** Adhyaay Karnwal  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)

## Setup

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3"

In [ ]:
from unsloth import FastLanguageModel
import torch
import json
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
MODEL_NAME = "unsloth/gemma-4-e2b-it"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8

## Load Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Verify LoRA attached correctly
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {all_params:,} ({100*trainable_params/all_params:.2f}%)")

## Load Dataset

Upload `train.json` and `val.json` from `data/processed/` as a Kaggle dataset.

In [ ]:
# Update this path to match your Kaggle dataset input
DATA_DIR = "/kaggle/input/datasets/adhyaaykarnwal/clover-cancer-data"

with open(os.path.join(DATA_DIR, "train.json")) as f:
    train_data = json.load(f)

with open(os.path.join(DATA_DIR, "val.json")) as f:
    val_data = json.load(f)

print(f"Train: {len(train_data)} examples")
print(f"Val: {len(val_data)} examples")
print(f"\nExample conversation:")
print(json.dumps(train_data[0]["conversations"][0], indent=2)[:200])

In [ ]:
from datasets import Dataset

def format_conversations(examples):
    texts = []
    for convo in examples["conversations"]:
        # Gemma 4 expects 'model' not 'assistant' as the role
        # Also merge system message into user message if present
        messages = []
        system_content = None
        for msg in convo:
            if msg["role"] == "system":
                system_content = msg["content"]
            elif msg["role"] == "user":
                content = msg["content"]
                if system_content:
                    content = f"{system_content}\n\n{content}"
                    system_content = None
                messages.append({"role": "user", "content": content})
            elif msg["role"] == "assistant":
                messages.append({"role": "model", "content": msg["content"]})

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset = train_dataset.map(format_conversations, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_conversations, batched=True, remove_columns=val_dataset.column_names)

# Debug: print a sample to verify format
print("Sample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])
print(f"\nContains '<start_of_turn>model\n': {'<start_of_turn>model' in train_dataset[0]['text']}")

print(f"\nTrain formatted: {len(train_dataset)} examples")
print(f"Val formatted: {len(val_dataset)} examples")

# CRITICAL: Verify the format before training
sample_text = train_dataset[0]["text"]

print("=" * 60)
print("FORMATTED TEXT SAMPLE (first 800 chars):")
print("=" * 60)
print(sample_text[:800])
print("=" * 60)

# Check for markers
has_user = "<start_of_turn>user" in sample_text
has_model = "<start_of_turn>model" in sample_text
print(f"\nContains '<start_of_turn>user': {has_user}")
print(f"Contains '<start_of_turn>model': {has_model}")

if not has_model:
    print("\nERROR: '<start_of_turn>model' marker NOT found!")
    print("This means the chat template isn't producing the expected format.")
    print("Full text repr:")
    print(repr(sample_text[:1000]))
else:
    print("\nMarkers found! Proceeding to training...")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.05,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        max_grad_norm=0.3,
        fp16=True,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Starting memory: {start_gpu_memory} GB")

In [ ]:
stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nTraining complete!")
print(f"  Steps: {stats.global_step}")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Time: {stats.metrics['train_runtime']:.0f}s")
print(f"  Peak memory: {used_memory}/{max_memory} GB ({100*used_memory/max_memory:.1f}%)")

## Save Model

In [ ]:
model.save_pretrained("clover-cancer-lora")
tokenizer.save_pretrained("clover-cancer-lora")
print("LoRA adapters saved.")

In [ ]:
model.save_pretrained_gguf(
    "clover-cancer-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF model saved.")

In [ ]:
TEST_SCENARIOS = [
    {
        "name": "Classic high-risk",
        "input": "I'm a 58-year-old male. I was just diagnosed with diabetes last month. I've lost about 15 pounds without trying. I have this deep ache in my mid-back that won't go away.",
        "expected": "high"
    },
    {
        "name": "Emergency jaundice",
        "input": "I'm a 67-year-old female. My skin turned yellow. I've lost about 10 pounds. My urine is really dark.",
        "expected": "critical"
    },
    {
        "name": "BRCA2 carrier",
        "input": "I'm a 52-year-old female with a BRCA2 mutation. My mother died of pancreatic cancer. I've been having abdominal pain and fatigue for 6 weeks.",
        "expected": "high"
    },
    {
        "name": "Low-risk back pain",
        "input": "I'm a 42-year-old male. I've had back pain for 2 weeks. I sit at a desk all day. No other symptoms.",
        "expected": "low"
    },
]

In [ ]:
SYSTEM_PROMPT = """You are a medical triage AI specialized in pancreatic cancer early detection.
Analyze symptoms and respond with JSON:
{"risk_assessment": "low|medium|high|critical", "conditions": [...], "urgency": "...", "recommended_actions": [...], "reasoning_chain": "..."}"""

FastLanguageModel.for_inference(model)

for scenario in TEST_SCENARIOS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Assess my symptoms:\n\n{scenario['input']}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"Scenario: {scenario['name']}")
    print(f"Expected risk: {scenario['expected']}")
    print(f"Response:\n{response}")

## Next Steps

1. Download the GGUF model from the output
2. Upload to Ollama or HuggingFace
3. Use in the Gradio demo
4. Record video and submit